# 04. Trajectory Discovery
ODE trajectory 동역학에서 새로운 응집 예측 신호 추출

**Direction A**: Folding Velocity Fingerprints — 후반부 velocity 수렴 지연 vs 응집
**Direction B**: Contact Kinetics — 접촉 형성/해체 불안정성 vs 응집
**Direction C**: Mutation Landscape — 전수 단일 돌연변이 스캔
**Integration**: 전체 피처 통합 상관분석 + LOPO CV

In [ ]:
# Setup
import os, sys
os.chdir('/content')
!rm -rf brain_idp_flow
!git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch
import numpy as np
import yaml
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# Load model + targets
import glob
from brain_idp_flow.targets import load_targets
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.sample import load_model, sample_ensemble_with_trajectory
from brain_idp_flow.data.dataset import ProteinEnsembleDataset

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

# Checkpoint — local or from Drive
ckpt_dirs = sorted(glob.glob('runs/flow/*'))
if ckpt_dirs:
    CKPT = os.path.join(ckpt_dirs[-1], 'ckpt_best.pt')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/brain_idp_flow_ckpts/ckpt_best.pt'

print(f'Checkpoint: {CKPT}')
model = load_model(config, CKPT, device)
targets = load_targets('configs/targets.yaml')
embedder = ESM2Embedder(device=device)

total_muts = sum(len(t.mutations) for t in targets.values())
print(f'Loaded {len(targets)} targets, {total_muts} mutations')

---
## Direction A + B: Trajectory Feature Extraction
28 mutations + 3 WT 에 대해 ODE trajectory 생성 → velocity + contact 피처 추출

In [ ]:
from brain_idp_flow.analysis.trajectory_analysis import extract_trajectory_features

os.makedirs('results/trajectory', exist_ok=True)

all_trajectory_results = []

for tid, target in targets.items():
    print(f'\n=== {target.name} ===')
    
    # WT trajectory
    wt_emb = embedder.embed_single(target.sequence)
    wt_result = sample_ensemble_with_trajectory(
        model, wt_emb,
        mut_pos=0, mut_aa=0,
        n_samples=200, n_trajectory_samples=16,
        device=device,
    )
    # WT trajectory features (pos=0 is arbitrary for WT)
    print(f'  WT ensemble: {wt_result["ensemble"].shape}')
    
    # Mutant trajectories
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut.mt, 0)
        
        mut_result = sample_ensemble_with_trajectory(
            model, mut_emb,
            mut_pos=mut.pos, mut_aa=aa_idx,
            n_samples=200, n_trajectory_samples=16,
            device=device,
        )
        
        # Extract trajectory features
        traj_feats = extract_trajectory_features(
            mut_result['trajectory'],
            mutation_pos=mut.pos - 1,  # 0-indexed
        )
        
        # Remove non-scalar fields for correlation
        result = {
            'target': tid,
            'mutation': mut.id,
            'pos': mut.pos,
            'agg_rate': mut.agg_rate_relative,
            **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
        }
        all_trajectory_results.append(result)
        
        print(f'  {mut.id}: late_vel={result["late_velocity_site"]:.4f}  '
              f'conv_t={result["convergence_time_site"]:.3f}  '
              f'switch={result["switching_rate_site"]:.4f}')

print(f'\nTotal: {len(all_trajectory_results)} mutation trajectory results')

In [ ]:
# Direction A: Velocity vs Aggregation correlation
from scipy.stats import spearmanr

agg_rates = np.array([r['agg_rate'] for r in all_trajectory_results])
log_agg = np.log(agg_rates + 1e-8)

velocity_features = [
    'late_velocity_site', 'late_velocity_global',
    'convergence_time_site', 'convergence_delay_vs_neighbors',
    'velocity_variance_late',
]

print('=== Direction A: Velocity Fingerprint Correlations ===')
print(f'{"Feature":<35} {"\u03c1":>8} {"p-value":>10} {"Sig?":>6}')
print('-' * 65)

for feat in velocity_features:
    vals = np.array([r[feat] for r in all_trajectory_results])
    rho, pval = spearmanr(vals, log_agg)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'{feat:<35} {rho:>8.3f} {pval:>10.4f} {sig:>6}')

In [ ]:
# Direction B: Contact Kinetics vs Aggregation
contact_features = [
    'switching_rate_site', 'switching_rate_long_range',
    'contact_order_site', 'early_contact_fraction_site',
]

print('=== Direction B: Contact Kinetics Correlations ===')
print(f'{"Feature":<35} {"\u03c1":>8} {"p-value":>10} {"Sig?":>6}')
print('-' * 65)

for feat in contact_features:
    vals = np.array([r[feat] for r in all_trajectory_results])
    rho, pval = spearmanr(vals, log_agg)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'{feat:<35} {rho:>8.3f} {pval:>10.4f} {sig:>6}')

In [ ]:
# Key Plot: Late-Stage Velocity vs Aggregation Rate
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

protein_colors = {'tau_K18': '#1f77b4', 'asyn': '#ff7f0e', 'abeta42': '#2ca02c'}
colors = [protein_colors[r['target']] for r in all_trajectory_results]
labels = [f"{r['target']}\n{r['mutation']}" for r in all_trajectory_results]

# Plot 1: Late velocity vs agg rate
ax = axes[0]
vals = [r['late_velocity_site'] for r in all_trajectory_results]
ax.scatter(vals, agg_rates, s=80, c=colors, edgecolors='black', linewidth=0.5)
for i, lbl in enumerate(labels):
    ax.annotate(lbl, (vals[i], agg_rates[i]), fontsize=5,
                textcoords='offset points', xytext=(5, 5))
rho, pval = spearmanr(vals, log_agg)
ax.set_xlabel('Late-Stage Velocity (site)')
ax.set_ylabel('Relative Aggregation Rate')
ax.set_title(f'Late Velocity vs Aggregation\n\u03c1={rho:.3f}, p={pval:.4f}')

# Plot 2: Convergence time vs agg rate
ax = axes[1]
vals = [r['convergence_time_site'] for r in all_trajectory_results]
ax.scatter(vals, agg_rates, s=80, c=colors, edgecolors='black', linewidth=0.5)
for i, lbl in enumerate(labels):
    ax.annotate(lbl, (vals[i], agg_rates[i]), fontsize=5,
                textcoords='offset points', xytext=(5, 5))
rho, pval = spearmanr(vals, log_agg)
ax.set_xlabel('Convergence Time (site)')
ax.set_ylabel('Relative Aggregation Rate')
ax.set_title(f'Convergence Time vs Aggregation\n\u03c1={rho:.3f}, p={pval:.4f}')

# Plot 3: Contact switching vs agg rate
ax = axes[2]
vals = [r['switching_rate_site'] for r in all_trajectory_results]
ax.scatter(vals, agg_rates, s=80, c=colors, edgecolors='black', linewidth=0.5)
for i, lbl in enumerate(labels):
    ax.annotate(lbl, (vals[i], agg_rates[i]), fontsize=5,
                textcoords='offset points', xytext=(5, 5))
rho, pval = spearmanr(vals, log_agg)
ax.set_xlabel('Contact Switching Rate (site)')
ax.set_ylabel('Relative Aggregation Rate')
ax.set_title(f'Contact Switching vs Aggregation\n\u03c1={rho:.3f}, p={pval:.4f}')

# Legend
for prot, color in protein_colors.items():
    axes[0].scatter([], [], c=color, label=prot, edgecolors='black', linewidth=0.5)
axes[0].legend(fontsize=9)

fig.tight_layout()
fig.savefig('results/trajectory/velocity_contact_vs_agg.png', dpi=150)
fig.show()

---
## Direction C: Mutation Landscape Scanning

In [ ]:
from brain_idp_flow.analysis.mutation_scanner import scan_full_landscape

os.makedirs('results/landscape', exist_ok=True)

landscape_results = {}

for tid, target in targets.items():
    result = scan_full_landscape(
        model, embedder, target, tid,
        top_n=50,
        n_ensemble=32,
        n_trajectory=4,
        n_steps=50,
        device=device,
        output_dir='results/landscape',
    )
    landscape_results[tid] = result
    
    # Top 10 novel predictions (not in known mutations)
    known_ids = {(m.pos, m.mt) for m in target.mutations}
    novel = [r for r in result['flow_results']
             if (r['pos'], r['mt']) not in known_ids]
    
    print(f'\n--- Top 5 novel high-risk mutations for {target.name} ---')
    for r in sorted(novel, key=lambda x: abs(x['delta_rg']), reverse=True)[:5]:
        print(f"  {r['wt']}{r['pos']}{r['mt']}: "
              f"LLR={r['llr_site']:.3f}, \u0394Rg={r['delta_rg']:.2f}, "
              f"late_vel={r.get('late_velocity_site', 0):.4f}")

In [ ]:
# Display landscape heatmaps
from IPython.display import Image, display

for tid in targets:
    img_path = f'results/landscape/{tid}_landscape.png'
    if os.path.exists(img_path):
        print(f'\n--- {tid} ---')
        display(Image(img_path))

---
## Integration: Full Correlation + LOPO CV

In [ ]:
# Merge trajectory features with ESM-2 + PED features
from brain_idp_flow.analysis.esm2_llr import score_all_mutations
from brain_idp_flow.analysis.ped_features import extract_all_ped_features

# ESM-2 LLR (full mode)
esm2_results = score_all_mutations(targets, device=device)

# PED structural features
os.makedirs('data/ped', exist_ok=True)
ped_results = extract_all_ped_features(targets, cache_dir='data/ped')

# Merge all three sources
merged = {}
for r in esm2_results:
    key = (r['target'], r['mutation'])
    merged[key] = {**r}
for r in ped_results:
    key = (r['target'], r['mutation'])
    if key in merged:
        merged[key].update({k: v for k, v in r.items()
                            if k not in ('target', 'mutation', 'pos', 'agg_rate')})
for r in all_trajectory_results:
    key = (r['target'], r['mutation'])
    if key in merged:
        merged[key].update({k: v for k, v in r.items()
                            if k not in ('target', 'mutation', 'pos', 'agg_rate')})

all_data = list(merged.values())
print(f'Merged data: {len(all_data)} mutations with all features')

In [ ]:
# Full correlation analysis with ALL features
from brain_idp_flow.analysis.aggregation_predictor import (
    run_correlation_analysis,
    leave_one_protein_out_cv,
)

os.makedirs('results/integrated', exist_ok=True)
summary = run_correlation_analysis(esm2_results, ped_results, output_dir='results/integrated')

# Re-run with trajectory features injected
# (The updated feature_names in aggregation_predictor.py now includes trajectory features)
summary_full = run_correlation_analysis(all_data, [], output_dir='results/integrated')

In [ ]:
# Leave-One-Protein-Out CV
lopo_results = leave_one_protein_out_cv(all_data)

print(f'\nLOPO Mean \u03c1: {lopo_results["mean_rho"]:.3f}')

In [ ]:
# Save all results + Drive backup
import json

final_results = {
    'trajectory_features': all_trajectory_results,
    'landscape': {
        tid: {
            'known_ranks': r['known_ranks'],
            'top_flow': r['flow_results'][:10],
        } for tid, r in landscape_results.items()
    },
    'lopo_cv': lopo_results,
}

with open('results/trajectory_discovery_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=float)

# Drive backup
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    import shutil
    backup = '/content/drive/MyDrive/brain_idp_flow_trajectory_discovery'
    shutil.copytree('results', backup, dirs_exist_ok=True)
    print(f'Backed up to {backup}')
except:
    print('Drive backup skipped')

print('\n=== TRAJECTORY DISCOVERY COMPLETE ===')